# Projet final : distillation

Ce notebook est une **amorce** : il fonctionne tel quel sur un réseau de démonstration. Votre
travail : l'appliquer à **votre** réseau du fil rouge, puis explorer. La consigne et les pistes :
<https://bonben.github.io/ensta-tp-deep-learning/05a_distillation/>.

Sur Colab : *Exécution > Modifier le type d'exécution > GPU T4*.

## Les données

In [ ]:
import os
import tarfile
import urllib.request

CIFAR_URL = "https://github.com/bonben/ensta-tp-deep-learning/releases/download/cifar10/cifar-10-python.tar.gz"

if not os.path.exists("data/cifar-10-batches-py"):
    urllib.request.urlretrieve(CIFAR_URL, "cifar.tar.gz")
    with tarfile.open("cifar.tar.gz") as tar:
        tar.extractall("data", filter="data")
    os.remove("cifar.tar.gz")
print("CIFAR-10 est prêt")

## Les outils

Ceux de la boîte à outils du fil rouge, plus `count_nonzero(model)`. `fit` accepte maintenant
une perte à vous (`loss_fn`), dont se sert la distillation.

In [ ]:
import copy
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision.transforms import v2
from torch.utils.flop_counter import FlopCounterMode
import matplotlib.pyplot as plt

torch.multiprocessing.set_start_method("fork", force=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Calcul sur :", device)

MEAN = (0.4914, 0.4822, 0.4465)
STD = (0.2470, 0.2435, 0.2616)
NORMALIZE = [v2.ToImage(), v2.ToDtype(torch.float32, scale=True), v2.Normalize(MEAN, STD)]

transform_train = v2.Compose([v2.RandomCrop(32, padding=4), v2.RandomHorizontalFlip()] + NORMALIZE)
transform_val = v2.Compose(NORMALIZE)
train_set = torchvision.datasets.CIFAR10("./data", train=True, download=True, transform=transform_train)
val_set = torchvision.datasets.CIFAR10("./data", train=False, download=True, transform=transform_val)
val_loader = torch.utils.data.DataLoader(val_set, batch_size=256, shuffle=False, num_workers=2)


def count_params(model):
    return sum(p.numel() for p in model.parameters())


def count_nonzero(model):
    """Paramètres non nuls (après élagage, les poids mis à zéro ne comptent plus)."""
    total = 0
    for m in model.modules():
        for name, p in m.named_parameters(recurse=False):
            if name.endswith("_orig"):  # poids élagué : on compte le poids masqué
                p = getattr(m, name[:-len("_orig")])
            total += int((p != 0).sum())
    return total


def count_macs(model):
    model.eval()
    with FlopCounterMode(display=False) as counter:
        model(torch.zeros(1, 3, 32, 32, device=next(model.parameters()).device))
    return counter.get_total_flops() // 2


def evaluate(model, loader=val_loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            correct += (model(inputs).argmax(1) == labels).sum().item()
            total += labels.size(0)
    return 100 * correct / total


def fit(model, epochs, lr=0.1, weight_decay=5e-4, batch_size=128, seed=0, name="essai", loss_fn=None):
    """Entraîne avec SGD (momentum 0.9) et un scheduler cosinus. Renvoie l'historique.

    loss_fn(outputs, labels, inputs) remplace la cross-entropy si on la donne (distillation).
    """
    torch.manual_seed(seed)
    train_loader = torch.utils.data.DataLoader(train_set, batch_size=batch_size, shuffle=True, num_workers=2)
    model = model.to(device)
    if loss_fn is None:
        loss_fn = lambda outputs, labels, inputs: F.cross_entropy(outputs, labels)
    optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    history = {"train_acc": [], "val_acc": []}
    for epoch in range(epochs):
        t0 = time.time()
        model.train()
        correct, total = 0, 0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = loss_fn(outputs, labels, inputs)
            loss.backward()
            optimizer.step()
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)
        scheduler.step()
        history["train_acc"].append(100 * correct / total)
        history["val_acc"].append(evaluate(model))
        print(f"époque {epoch + 1}/{epochs} | train {history['train_acc'][-1]:.1f} % | "
              f"val {history['val_acc'][-1]:.1f} % | {time.time() - t0:.0f} s")
    print(f"\nRÉSUMÉ : {name} | {count_params(model)} paramètres ({count_nonzero(model)} non nuls) | {epochs} époques | "
          f"val au dernier epoch {history['val_acc'][-1]:.2f} %")
    return history


def plot(*histories, labels=None):
    """Superpose les courbes de validation de plusieurs essais."""
    for i, h in enumerate(histories):
        plt.plot(h["val_acc"], label=labels[i] if labels else f"essai {i + 1}")
    plt.xlabel("époque"); plt.ylabel("précision de validation (%)")
    plt.grid(alpha=0.3); plt.legend(); plt.show()

## Le réseau

Le ResNet du fil rouge et sa version factorisée (cours 3). **Remplacez ou complétez cette cellule
par la classe de votre réseau.**

In [ ]:
class BasicBlock(nn.Module):
    """Deux convolutions 3×3 et un raccourci (la connexion résiduelle)."""

    def __init__(self, in_planes, planes, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, planes, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(planes),
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = out + self.shortcut(x)
        return F.relu(out)


class SeparableConv2d(nn.Module):
    """Convolution séparable : depthwise 3×3 puis pointwise 1×1 (cours 3)."""

    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.depthwise = nn.Conv2d(in_ch, in_ch, 3, stride=stride, padding=1, groups=in_ch, bias=False)
        self.pointwise = nn.Conv2d(in_ch, out_ch, 1, bias=False)

    def forward(self, x):
        return self.pointwise(self.depthwise(x))


class SepBasicBlock(BasicBlock):
    def __init__(self, in_planes, planes, stride=1):
        super().__init__(in_planes, planes, stride)
        self.conv1 = SeparableConv2d(in_planes, planes, stride)
        self.conv2 = SeparableConv2d(planes, planes, 1)


class ResNet(nn.Module):
    """4 étages ; le nombre de FM vaut width, 2·width, 4·width, 8·width."""

    def __init__(self, block, num_blocks, num_classes=10, width=64):
        super().__init__()
        self.in_planes = width
        self.conv1 = nn.Conv2d(3, width, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(width)
        self.layer1 = self._make_layer(block, width, num_blocks[0], stride=1)
        self.layer2 = self._make_layer(block, 2 * width, num_blocks[1], stride=2)
        self.layer3 = self._make_layer(block, 4 * width, num_blocks[2], stride=2)
        self.layer4 = self._make_layer(block, 8 * width, num_blocks[3], stride=2)
        self.linear = nn.Linear(8 * width, num_classes)

    def _make_layer(self, block, planes, num_blocks, stride):
        layers = []
        for s in [stride] + [1] * (num_blocks - 1):
            layers.append(block(self.in_planes, planes, s))
            self.in_planes = planes
        return nn.Sequential(*layers)

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)
        out = F.adaptive_avg_pool2d(out, 1).flatten(1)
        return self.linear(out)

## Votre réseau de départ

Par défaut, le réseau de démonstration : le ResNet18 factorisé, `width=32` (371 658 paramètres,
environ 89 % en 15 époques). Mettez le vôtre à la place, et rechargez ses poids si vous les avez
sauvegardés : pas besoin de le réentraîner.

In [ ]:
model = ResNet(SepBasicBlock, [2, 2, 2, 2], width=32).to(device)   # <- votre réseau
print(f"{count_params(model):,} paramètres, {count_macs(model):,} MAC")

# Vos poids sauvegardés dans Drive (décommentez) :
# from google.colab import drive
# drive.mount("/content/drive")
# model.load_state_dict(torch.load("/content/drive/MyDrive/mon_reseau.pt", map_location=device))
# print(f"précision : {evaluate(model):.2f} %")

In [ ]:
# Sinon, entraînez-le ici (une fois), puis sauvegardez-le.
h_ref = fit(model, epochs=15, name="réseau de départ")
torch.save(model.state_dict(), "reseau_depart.pt")

## L'enseignant

Un ResNet18 de largeur 32 (2,8 M paramètres, 94,9 %), entraîné 200 époques avec une augmentation
forte (TrivialAugment). On le télécharge : inutile de l'entraîner.

In [ ]:
TEACHER_URL = "https://github.com/bonben/ensta-tp-deep-learning/releases/download/cifar10/teacher_resnet18_w32.pt"

teacher = ResNet(BasicBlock, [2, 2, 2, 2], width=32)
if not os.path.exists("teacher_resnet18_w32.pt"):
    urllib.request.urlretrieve(TEACHER_URL, "teacher_resnet18_w32.pt")
teacher.load_state_dict(torch.load("teacher_resnet18_w32.pt", map_location="cpu"))
teacher = teacher.to(device).eval()
print(f"Enseignant : {count_params(teacher):,} paramètres, {evaluate(teacher):.2f} % en validation")

## La perte de distillation

L'élève apprend à reproduire les **sorties adoucies** de l'enseignant (softmax à température
`T`), et un peu les vraies étiquettes. `alpha` dose les deux.

In [ ]:
def kd_loss(teacher, T=4.0, alpha=0.9):
    """La perte de Hinton : imiter l'enseignant (sorties adoucies par T) et les vraies étiquettes."""

    def loss_fn(outputs, labels, inputs):
        with torch.no_grad():
            teacher_outputs = teacher(inputs)
        soft = F.kl_div(F.log_softmax(outputs / T, dim=1), F.softmax(teacher_outputs / T, dim=1),
                        reduction="batchmean") * T * T
        hard = F.cross_entropy(outputs, labels)
        return alpha * soft + (1 - alpha) * hard

    return loss_fn

## Un premier essai

Un élève **de la même architecture** que votre réseau de départ, entraîné depuis zéro avec la
même recette et le même nombre d'époques, mais avec la perte de distillation. Seule la perte
change : la comparaison est honnête. Chaque époque coûte un peu plus cher (l'enseignant calcule
aussi).

In [ ]:
student = ResNet(SepBasicBlock, [2, 2, 2, 2], width=32)   # <- la même architecture que votre réseau
h_kd = fit(student, epochs=15, loss_fn=kd_loss(teacher, T=4.0, alpha=0.9), name="élève distillé")
plot(h_ref, h_kd, labels=["sans distillation", "avec distillation"])

## À vous

Une question à la fois, en comparant à nombre d'époques égal. Quelques pistes :

- la **température** `T` (1, 2, 4, 8) et le dosage `alpha` ;
- un **élève plus petit** : jusqu'où la distillation compense-t-elle la taille perdue ?
- **plus d'époques** : l'écart avec l'entraînement normal se maintient-il ?
- un autre enseignant : votre meilleur réseau du fil rouge.

In [ ]:
# student = ...
# h = fit(student, epochs=15, loss_fn=kd_loss(teacher, T=..., alpha=...), name="...")